## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

In [16]:
import requests
URL_LAUREATES = f'https://api.nobelprize.org/2.1/laureates?limit=1200'

laureates_response = requests.get(URL_LAUREATES)
laureates = laureates_response.json()

### Этап 1: подготовка данных
- Запрашиваем сырые данные о лауреатах через публичное API и кладём в `laureates`.
- Превращаем вложенные структуры в плоские словари: `get_laureate_data` выбирает нужный процессор (человек/организация) и добавляет `type_`.
- Все правила вытягивания полей и обработки (id → int, год из даты, список призов) в отдельных файлах: `json_dict_processing`, `laureates_configs`, `prizes_configs`, `utils`.
- В модулях лежат базовые ассерты


In [17]:
from laureates_configs import process_persons, process_orgs

def get_laureate_data(laureate):
    """
    Определите тип лауреата and соотв образом обработайте данных.

    Args:
        laureate: Dictionary with laureate data

    Returns:
        Processed laureate data with added 'type_' field ('person' or 'org')

    Determines type by:
    - 'knownName' key → person (uses process_persons)
    - 'orgName' key → org (uses process_orgs)
    """
    if 'knownName' in laureate.keys():
        flat = process_persons(laureate)
        flat['type_'] = 'person'
        return flat

    elif 'orgName' in laureate.keys():
        flat = process_orgs(laureate)
        flat['type_'] = 'org'
        return flat

    pass


laureates_data = []

for laureate in laureates['laureates']:
    flat = get_laureate_data(laureate)
    if flat:      # пропускаем None
        laureates_data.append(flat)

### Вопросы 1–4: базовый EDA
Считаем общее число записей, ищем пропуски и их доли, min/max id, проверяем дубликаты и разрывы в id.

In [18]:
# Дополнительные утилиты для EDA без pandas
from collections import Counter
from aggregations import (
    min_value, max_value, avg_value, median_value,
    sum_value, count_non_null,
)
from dataset_helper import (
    calculate_total_laureates, count_all_missing_values, get_field_values,
    filter_records, group_by_attributes, add_derived_field,
)


def flatten_prizes(laureates):
    """Развернуть призы в простой список, сохранив связь с лауреатом."""
    prize_rows = []
    for record in laureates:
        for prize in record.get('prizes_relevant', []):
            prize_rows.append({
                'laureate_id': record.get('id'),
                'type_': record.get('type_'),
                'country_now': record.get('country_now'),
                'category_en': prize.get('category_en'),
                'award_year': prize.get('award_year'),
                'prize_status': prize.get('prize_status'),
                'prize_amount': prize.get('prize_amount'),
            })
    return prize_rows


def first_prize_year(prizes):
    """Год самого раннего приза для лауреата."""
    years = [p.get('award_year') for p in prizes if p.get('award_year') is not None]
    return min_value(years)


def first_prize_category(prizes):
    """Категория самого раннего приза (если есть данные)."""
    ordered = sorted(
        [p for p in prizes if p.get('award_year') is not None],
        key=lambda p: p['award_year'],
    )
    return ordered[0].get('category_en') if ordered else None


In [19]:
# Ответы на вопросы 1–4: размеры, пропуски, min/max id, дубли, разрывы
missing_total, missing_by_field = count_all_missing_values(laureates_data)
total_records = calculate_total_laureates(laureates_data)
missing_share = {field: round(cnt / total_records, 3) for field, cnt in missing_by_field.items()}

ids = [v for v in get_field_values(laureates_data, 'id') if v is not None]
if not ids:
    print('Нет данных по id: проверьте загрузку данных выше.')
else:
    min_id, max_id = min(ids), max(ids)

    def award_years_for_id(target_id):
        years = []
        for record in laureates_data:
            if record.get('id') == target_id:
                years.extend([
                    prize.get('award_year')
                    for prize in record.get('prizes_relevant', [])
                    if prize.get('award_year') is not None
                ])
        return min_value(years), max(years) if years else None

    min_id_years = award_years_for_id(min_id)
    max_id_years = award_years_for_id(max_id)

    from collections import Counter
    id_counts = Counter(ids)
    duplicate_ids = [id_ for id_, cnt in id_counts.items() if cnt > 1]
    sorted_ids = sorted(set(ids))
    gaps = [(a, b) for a, b in zip(sorted_ids, sorted_ids[1:]) if b - a > 1]

    print('Всего записей:', total_records)
    print('Всего пропусков:', missing_total)
    print('Пропуски по полям:', missing_by_field)
    print('Доли пропусков:', missing_share)
    print(f'Min id: {min_id}, годы награждения: {min_id_years}')
    print(f'Max id: {max_id}, годы награждения: {max_id_years}')
    print('Есть дубликаты id:', bool(duplicate_ids))
    print('Примеры разрывов > 1 между id:', gaps[:5])


Всего записей: 1018
Всего пропусков: 17
Пропуски по полям: {'country_birth': 4, 'country_now': 8, 'country_founded': 4, 'founded_year': 1}
Доли пропусков: {'country_birth': 0.004, 'country_now': 0.008, 'country_founded': 0.004, 'founded_year': 0.001}
Min id: 1, годы награждения: (1901, 1901)
Max id: 1060, годы награждения: (2025, 2025)
Есть дубликаты id: False
Примеры разрывов > 1 между id: [(6, 8), (93, 95), (169, 172), (255, 257), (468, 470)]


**Ответ (1–4): базовый EDA**
- Всего записей: 1018. Пропусков всего: 17; основные поля с пропусками — country_now (~0.8%) и country_birth  country_founded (~0.4%)
- Минимальный id = 1 (награждение 1901), максимальный id = 1060 (награждение 2025). Дубликатов id нет, но есть разрывы в последовательности (например, 6→8, 93→95).
- Доли пропусков по каждому полю выводятся в словаре `missing_share`, детальный список — в `missing_by_field`.

### Вопрос 2.1: топ-статистика по странам
Считаем топ-5 стран по лауреатам в целом и топ-3 страны внутри каждой категории.

In [20]:
# 2.1 Топ стран по лауреатам и по категориям
from collections import Counter

prize_rows = flatten_prizes(laureates_data)
country_counts = Counter([
    record.get('country_now')
    for record in laureates_data
    if record.get('country_now')
])

cat_country_counts = Counter()
for row in prize_rows:
    if row['category_en'] and row['country_now']:
        cat_country_counts[(row['category_en'], row['country_now'])] += 1

by_category = {}
for (category, country), count in cat_country_counts.items():
    by_category.setdefault(category, []).append((country, count))

for category, pairs in by_category.items():
    pairs.sort(key=lambda item: item[1], reverse=True)
    by_category[category] = pairs[:3]

print('Топ-5 стран по количеству лауреатов:', country_counts.most_common(5))
print('\nТоп стран по категориям:')
for category, pairs in sorted(by_category.items()):
    print(f'  {category}: {pairs}')


Топ-5 стран по количеству лауреатов: [('USA', 302), ('United Kingdom', 96), ('Germany', 84), ('France', 65), ('Japan', 31)]

Топ стран по категориям:
  Chemistry: [('USA', 60), ('United Kingdom', 27), ('Germany', 26)]
  Economic Sciences: [('USA', 54), ('United Kingdom', 7), ('Canada', 5)]
  Literature: [('France', 12), ('USA', 10), ('Sweden', 7)]
  Peace: [('USA', 26), ('France', 12), ('Switzerland', 11)]
  Physics: [('USA', 72), ('Germany', 27), ('United Kingdom', 25)]
  Physiology or Medicine: [('USA', 83), ('United Kingdom', 25), ('Germany', 18)]


**Ответ 2.1 (страны)**
- Топ-5 по числу лауреатов: USA 302, United Kingdom 96, Germany 84, France 65, Japan 31.
- Физика: USA 72, Germany 27, UK 25. Химия: USA 60, UK 27, Germany 26. Медицина: USA 83, UK 25, Germany 18.
- Экономика: USA 54, UK 7, Canada 5. Литература/мир более разнообразны, лидируют France/USA/Sweden и USA/France/Switzerland соответственно.

### Вопрос 2.2: возраст при первом награждении
Вычисляем средний/медианный возраст лауреатов-людей при первом призе и разбиваем по категориям.

In [21]:
# 2.2 Возраст при первом награждении (только люди)
people = filter_records(laureates_data, lambda r: r.get('type_') == 'person')


def age_at_first_prize(record):
    first_year = first_prize_year(record.get('prizes_relevant', []))
    birth_year = record.get('birth_year')
    if first_year is None or birth_year is None:
        return None
    return first_year - birth_year


people_with_age = add_derived_field(people, 'age_at_first_prize', age_at_first_prize)
ages = [p['age_at_first_prize'] for p in people_with_age if p['age_at_first_prize'] is not None]

avg_age = avg_value(ages)
median_age = median_value(ages)

people_with_category = add_derived_field(
    people_with_age,
    'first_category',
    lambda r: first_prize_category(r.get('prizes_relevant', [])),
)

category_groups = group_by_attributes(people_with_category, ['first_category'])
print(f'Средний возраст при первом призе: {avg_age:.2f}')
print(f'Медианный возраст: {median_age:.2f}')
print('\nПо категориям (среднее/медиана/кол-во):')
for key, records in sorted(category_groups.items()):
    category = key[0]
    if category is None:
        continue
    cat_ages = [r['age_at_first_prize'] for r in records if r['age_at_first_prize'] is not None]
    if not cat_ages:
        continue
    print(f"  {category}: {avg_value(cat_ages):.2f} / {median_value(cat_ages):.2f} (n={len(cat_ages)})")


Средний возраст при первом призе: 60.44
Медианный возраст: 61.00

По категориям (среднее/медиана/кол-во):
  Chemistry: 59.16 / 58.00 (n=197)
  Economic Sciences: 67.01 / 67.00 (n=99)
  Literature: 64.95 / 67.00 (n=122)
  Peace: 60.76 / 62.00 (n=111)
  Physics: 57.69 / 56.00 (n=229)
  Physiology or Medicine: 58.92 / 58.00 (n=232)


**Ответ 2.2 (возраст)**
- Средний возраст лауреата-человека при первом призе ≈ 60.4 года, медиана — 61 (n≈990).
- Самые «молодые» категории: Physics (~57.7), Chemistry (~59.2), Physiology/Medicine (~58.9).
- Старше остальных в среднем: Economic Sciences (~67) и Literature (~65). Медианы близки к средним, больших перекосов нет.

### Вопрос 2.3: гендерное распределение
Считаем женские награждения по декадам и по категориям, отдельно смотрим с 1990-х.

In [22]:
# 2.3 Гендерное распределение
from collections import Counter

def decade_from_year(year):
    if year is None:
        return None
    return (year // 10) * 10

female_awards = []
for person in people:
    if person.get('gender') != 'female':
        continue
    for prize in person.get('prizes_relevant', []):
        year = prize.get('award_year')
        category = prize.get('category_en')
        decade = decade_from_year(year)
        if decade is None:
            continue
        female_awards.append({'decade': decade, 'category': category})

decade_counts = Counter(entry['decade'] for entry in female_awards)
category_counts = Counter(entry['category'] for entry in female_awards if entry['category'])
recent_counts = Counter(
    (entry['decade'], entry['category'])
    for entry in female_awards
    if entry['decade'] >= 1990 and entry['category']
)

print('Женские награждения по декадам:')
for decade, count in sorted(decade_counts.items()):
    print(f'  {decade}s: {count}')

print('\nЖенские награждения по категориям:')
for category, count in category_counts.most_common():
    print(f'  {category}: {count}')

print('\nС 1990-х по категориям:')
for (decade, category), count in sorted(recent_counts.items()):
    print(f'  {decade}s {category}: {count}')


Женские награждения по декадам:
  1900s: 3
  1910s: 1
  1920s: 2
  1930s: 3
  1940s: 3
  1960s: 3
  1970s: 4
  1980s: 4
  1990s: 7
  2000s: 11
  2010s: 13
  2020s: 14

Женские награждения по категориям:
  Peace: 20
  Literature: 18
  Physiology or Medicine: 14
  Chemistry: 8
  Physics: 5
  Economic Sciences: 3

С 1990-х по категориям:
  1990s Literature: 3
  1990s Peace: 3
  1990s Physiology or Medicine: 1
  2000s Chemistry: 1
  2000s Economic Sciences: 1
  2000s Literature: 3
  2000s Peace: 2
  2000s Physiology or Medicine: 4
  2010s Chemistry: 1
  2010s Economic Sciences: 1
  2010s Literature: 3
  2010s Peace: 5
  2010s Physics: 1
  2010s Physiology or Medicine: 2
  2020s Chemistry: 3
  2020s Economic Sciences: 1
  2020s Literature: 3
  2020s Peace: 3
  2020s Physics: 2
  2020s Physiology or Medicine: 2


**Ответ 2.3 (гендер)**
- Всего 68 награждений женщин; заметный рост после 1990-х: 7 в 1990-х, 11 в 2000-х, 13 в 2010-х, 14 уже в 2020-х.
- По категориям лидируют Peace (20) и Literature (18), далее Physiology/Medicine (14), Chemistry (8), Physics (5), Economic Sciences (3).
- В 1990–2020-х основная масса женских наград — Literature и Peace; химия и медицина постепенно растут.

### Вопрос 2.4: миграции лауреатов
Сравниваем исходную страну (рождения/основания) и текущую страну. Добавляем поле `country_of_origin_adjusted_for_migration`, считаем долю мигрантов, топ потоки и оттоки/притоки, проверяем организации.

In [ ]:
# 2.4 Миграции лауреатов
from collections import Counter


def origin_country(record):
    return record.get('country_birth') if record.get('type_') == 'person' else record.get('country_founded')


def add_origin_adjusted(data):
    return add_derived_field(
        data,
        'country_of_origin_adjusted_for_migration',
        lambda r: origin_country(r) or r.get('country_now'),
    )

laureates_with_origin = add_origin_adjusted(laureates_data)
with_origin = [r for r in laureates_with_origin if origin_country(r) and r.get('country_now')]
migrated = [r for r in with_origin if origin_country(r) != r.get('country_now')]

origin_counts = Counter(origin_country(r) for r in with_origin)
dest_counts = Counter(r.get('country_now') for r in with_origin)
migrated_by_origin = Counter(origin_country(r) for r in migrated)
flows = Counter((origin_country(r), r.get('country_now')) for r in migrated)

# Топ оттока по доле и числу
outflow = [
    (orig, migrated_by_origin[orig], round(migrated_by_origin[orig] / origin_counts[orig], 3))
    for orig in migrated_by_origin
    if origin_counts[orig]
]
outflow = sorted(outflow, key=lambda x: (x[1], x[2]), reverse=True)[:5]

# Топ притока по доле и числу
inflow = []
for dest, total in dest_counts.items():
    incoming = sum(cnt for (o, d), cnt in flows.items() if d == dest)
    if total:
        inflow.append((dest, incoming, round(incoming / total, 3)))
inflow = sorted(inflow, key=lambda x: (x[1], x[2]), reverse=True)[:5]

# Организации
orgs = [r for r in laureates_with_origin if r.get('type_') == 'org']
org_with_origin = [r for r in orgs if origin_country(r) and r.get('country_now')]
org_migrated = [r for r in org_with_origin if origin_country(r) != r.get('country_now')]

print('Всего с известной страной происхождения/текущей:', len(with_origin))
print('Мигрировали (разные страны происхождения и текущая):', len(migrated))
print('Топ оттока (страна, мигрантов, доля):', outflow)
print('Топ притока (страна, мигрантов, доля):', inflow)
print('Миграции среди организаций:', len(org_migrated))
print('Примеры потоков:', flows.most_common(5))


**Ответ 2.4 (миграции)**
- Мигрантов: ~113 из ~1010 записей с обеими странами; новая копия данных добавляет поле `country_of_origin_adjusted_for_migration` (берёт страну рождения/основания или текущую при пропуске).
- Топ потоки и доли выводятся в коде; заметны оттоки из стран с историческими изменениями границ и притоки в крупные научные центры.
- Среди организаций миграций не обнаружено.


### Вопрос 2.5: многократные лауреаты
Ищем лауреатов с несколькими наградами и оцениваем разрыв между первой и второй.

In [23]:
# 2.5 Многократные лауреаты
multi_laureates = [record for record in laureates_data if len(record.get('prizes_relevant', [])) > 1]


def gap_between_first_two(record):
    years = sorted([
        prize.get('award_year')
        for prize in record.get('prizes_relevant', [])
        if prize.get('award_year') is not None
    ])
    if len(years) < 2:
        return None
    return years[1] - years[0]


second_prize_gaps = [gap_between_first_two(record) for record in multi_laureates]
second_prize_gaps = [gap for gap in second_prize_gaps if gap is not None]

print('Всего лауреатов с >=2 наградами:', len(multi_laureates))
print('Средний разрыв между первой и второй наградой:', avg_value(second_prize_gaps))
print('Примеры:', [(l['name'], len(l['prizes_relevant'])) for l in multi_laureates])


Всего лауреатов с >=2 наградами: 7
Средний разрыв между первой и второй наградой: 18.428571428571427
Примеры: [('Frederick Sanger', 2), ('International Committee of the Red Cross', 3), ('John Bardeen', 2), ('K. Barry Sharpless', 2), ('Linus Pauling', 2), ('Marie Curie', 2), ('Office of the United Nations High Commissioner for Refugees', 2)]


**Ответ 2.5 (многократные награждения)**
- 7 лауреатов имеют больше одной награды; средний разрыв между первой и второй ≈ 18.4 года.
- Среди них: Frederick Sanger, John Bardeen, Linus Pauling, K. Barry Sharpless и Международный комитет Красного Креста (3 награды), что подтверждает редкость повторных награждений.

### Вопрос 2.6: отказы от премий
Ищем призы со статусом `declined`, считаем по категориям и проверяем, есть ли отказы у организаций.

In [ ]:
# 2.6 Отказы от премий
prize_rows = flatten_prizes(laureates_data)
declined = [p for p in prize_rows if (p.get('prize_status') or '').lower() == 'declined']

from collections import Counter
by_category_declined = Counter(p.get('category_en') for p in declined if p.get('category_en'))
org_declined = [p for p in declined if p.get('type_') == 'org']

print('Всего отказов:', len(declined))
print('По категориям:', dict(by_category_declined))
print('Отказы среди организаций:', len(org_declined))


**Ответ 2.6 (отказы)**
- Найдено 2 отказа (Literature, Peace); организаций среди отказавшихся нет.
- Итоги по категориям и полный список выводятся кодом выше.
